# Assignment 5 
**Name:** Bhanavi
**Roll No:** 102313054

### Q1 
(Based on Step-by-Step Implementation of Ridge Regression using Gradient
Descent Optimization) Generate a dataset with atleast seven highly correlated columns and a target variable. Implement Ridge Regression using Gradient Descent Optimization. Take different values of learning rate (such as 0.0001,0.001,0.01,0.1,1,10) and regularization parameter (10-15,10-10,10-5,10-3,0,1,10,20). Choose the best parameters for which ridge regression cost function is minimum and R2_score is maximum.

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

np.random.seed(42)
n_samples = 500
X_base = np.random.randn(n_samples, 1)

X = np.hstack([X_base + 0.1*np.random.randn(n_samples,1) for _ in range(7)])

true_w = np.array([2, -3, 1.5, 0.5, -1, 2.5, -0.7])
y = X.dot(true_w) + 0.2*np.random.randn(n_samples)
X_b = np.hstack([np.ones((n_samples,1)), X])

def ridge_gradient_descent(X, y, lr, lam, iterations=2000):
    m, n = X.shape
    w = np.zeros(n)

    for _ in range(iterations):
        y_pred = X.dot(w)
        grad = -(2/m) * X.T.dot(y - y_pred) + 2 * lam * w
        w_new = w - lr * grad

        if not np.all(np.isfinite(w_new)):
            return None

        w = w_new

    return w

learning_rates = [0.0001, 0.001, 0.01, 0.1, 1, 10]
lambdas = [1e-15, 1e-10, 1e-5, 1e-3, 0, 1, 10, 20]

results = []

for lr in learning_rates:
    for lam in lambdas:
        w = ridge_gradient_descent(X_b, y, lr, lam)
        if w is None:
            continue
        
        y_pred = X_b.dot(w)
        cost = np.mean((y - y_pred)**2) + lam * np.sum(w**2)
        r2 = r2_score(y, y_pred)
        
        results.append((lr, lam, cost, r2))

best = sorted(results, key=lambda x: (-x[3], x[2]))[0]

print("\nBest parameters found:")
print(f"Learning rate (lr): {best[0]}")
print(f"Regularization (lambda): {best[1]}")
print(f"Cost: {best[2]}")
print(f"R2 Score: {best[3]}")



Best parameters found:
Learning rate (lr): 0.1
Regularization (lambda): 0
Cost: 0.03853512896137298
R2 Score: 0.9887293935511514


C:\Users\bhana\AppData\Local\Temp\ipykernel_20844\2720865877.py:26: RuntimeWarning: overflow encountered in multiply
  grad = -(2/m) * X.T.dot(y - y_pred) + 2 * lam * w


### Q2
Load the Hitters dataset from the following link: https://drive.google.com/file/d/1qzCKF6JKKMB0p7ul_1Ly8tdmRk3vE_bG/view?usp=sharing
(a) Pre-process the data (null values, noise, categorical to numerical encoding) 
(b) Separate input and output features and perform scaling 
(c) Fit a Linear, Ridge (use regularization parameter as 0.5748), and LASSO (use regularization parameter as 0.5748) regression function on the dataset.
(d) Evaluate the performance of each trained model on test set. Which model performs the best and Why?

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error


df = pd.read_csv("Hitters.csv")  

print("Missing values:\n", df.isnull().sum())
df = df.dropna()
categorical_cols = ['League', 'Division', 'NewLeague']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df.drop("Salary", axis=1)
y = df["Salary"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
# Ridge Regression (lambda = alpha)
ridge_reg = Ridge(alpha=0.5748)
ridge_reg.fit(X_train, y_train)
# Lasso Regression (lambda = alpha)
lasso_reg = Lasso(alpha=0.5748, max_iter=5000)
lasso_reg.fit(X_train, y_train)

models = {
    "Linear Regression": lin_reg,
    "Ridge Regression": ridge_reg,
    "Lasso Regression": lasso_reg
}

print("\nModel Performance on Test Set")

for name, model in models.items():
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"{name}: R2 = {r2:.4f}, MSE = {mse:.4f}")


Missing values:
 AtBat         0
Hits          0
HmRun         0
Runs          0
RBI           0
Walks         0
Years         0
CAtBat        0
CHits         0
CHmRun        0
CRuns         0
CRBI          0
CWalks        0
League        0
Division      0
PutOuts       0
Assists       0
Errors        0
Salary       59
NewLeague     0
dtype: int64

Model Performance on Test Set
Linear Regression: R2 = 0.2907, MSE = 128284.3455
Ridge Regression: R2 = 0.2998, MSE = 126648.5942
Lasso Regression: R2 = 0.2994, MSE = 126714.5168


Ridge Regression performs best with highest R² score.

### Q3
Cross Validation for Ridge and Lasso Regression. Explore Ridge Cross Validation (RidgeCV) and Lasso Cross Validation (LassoCV)function of Python. Implement both on Boston House Prediction Dataset (load_boston
dataset from sklearn.datasets).

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_squared_error

data = fetch_california_housing()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# RidgeCV (Cross Validation)
ridge_alphas = np.logspace(-3, 3, 50)  
ridge_cv = RidgeCV(alphas=ridge_alphas, cv=5)
ridge_cv.fit(X_train, y_train)

ridge_pred = ridge_cv.predict(X_test)

print("\nRidgeCV Results")
print("Best alpha:", ridge_cv.alpha_)
print("R2 Score:", r2_score(y_test, ridge_pred))
print("MSE:", mean_squared_error(y_test, ridge_pred))

# LassoCV (Cross Validation)
lasso_cv = LassoCV(cv=5, max_iter=5000)
lasso_cv.fit(X_train, y_train)

lasso_pred = lasso_cv.predict(X_test)

print("\nLassoCV Results")
print("Best alpha:", lasso_cv.alpha_)
print("R2 Score:", r2_score(y_test, lasso_pred))
print("MSE:", mean_squared_error(y_test, lasso_pred))



RidgeCV Results
Best alpha: 0.001
R2 Score: 0.5757877341612516
MSE: 0.5558915618350112

LassoCV Results
Best alpha: 0.000798519564426035
R2 Score: 0.5766495309609692
MSE: 0.554762255571242


### Q4
Multiclass Logistic Regression: Implement Multiclass Logistic Regression (step-by step) on Iris dataset using one vs. rest strategy?

In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data           
y = iris.target         

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LogisticRegression(multi_class='ovr', max_iter=200)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 0.9666666666666667


c:\Users\bhana\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
